# Home Credit auxiliary Spark parity — local smoke

Build one candidate block at a time. This notebook does not train LightGBM, create a submission, overwrite a reference block, or promote a candidate.

In [ ]:
from __future__ import annotations

import gc
import json
import os
import platform
import sys
import time
from pathlib import Path

EXPECTED_PYTHON = r'C:\\Python310\\python.exe'
assert sys.executable.lower() == EXPECTED_PYTHON.lower(), (
    f'Use the C:/Python310 kernel; got {sys.executable!r}.'
)
os.environ['PYSPARK_PYTHON'] = EXPECTED_PYTHON
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

DATA_DIR = Path(os.environ['HOME_CREDIT_DATA_DIR'])
FEATURE_STORE = Path(os.environ['HOME_CREDIT_FEATURE_STORE'])
CANDIDATE_ROOT = FEATURE_STORE / 'Spark'
OUTPUT_ROOT = Path(os.environ['HOME_CREDIT_OUTPUT_DIR']) / 'spark_auxiliary_parity'
assert DATA_DIR.is_dir(), f'Missing HOME_CREDIT_DATA_DIR: {DATA_DIR}'
CANDIDATE_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.master('local[2]').appName('home-credit-auxiliary-parity')
    .config('spark.sql.shuffle.partitions', '16')
    .config('spark.default.parallelism', '16')
    .config('spark.pyspark.python', os.environ['PYSPARK_PYTHON'])
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print({'python': sys.executable, 'spark': spark.version, 'platform': platform.platform()})


In [ ]:
# Run exactly one entry at a time; attach pandas reference blocks under FEATURE_STORE.
from credit_scoring.features.home_credit_previous_application_spark import (
    build_previous_application_features_spark, previous_application_csv_schema_spark,
    write_previous_application_feature_block_spark,
)

RUN = 'previous_application'  # then installments, credit_card, pos_cash
if RUN != 'previous_application':
    raise ValueError('Select one configured block and run it independently.')

started = time.perf_counter()
raw = (spark.read.option('header', True).schema(previous_application_csv_schema_spark())
       .csv(str(DATA_DIR / 'previous_application.csv')))
read_seconds = time.perf_counter() - started
started = time.perf_counter()
candidate = build_previous_application_features_spark(raw)
row_count = candidate.count()
aggregation_seconds = time.perf_counter() - started
manifest = write_previous_application_feature_block_spark(candidate, root=CANDIDATE_ROOT)
diagnostics = {'block': RUN, 'raw_csv_read_seconds': read_seconds, 'aggregation_seconds': aggregation_seconds, 'row_count': row_count, 'manifest': manifest.to_dict()}
(OUTPUT_ROOT / f'{RUN}_runtime_summary.json').write_text(json.dumps(diagnostics, indent=2), encoding='utf-8')
raw.unpersist(blocking=False)
candidate.unpersist(blocking=False)
gc.collect()
